# Supervised Fine-Tuning (SFT) without GPU training: import a pre-trained adapter

## Lab 2b - Import pre-trained adapter (no GPU needed)

GPU capacity for Lab 2 can be hard to get. This notebook is the no-GPU alternative to [2-supervised-fine-tuning.ipynb](./2-supervised-fine-tuning.ipynb): it takes the LoRA adapter in [pretrained/](./pretrained/README.md), trained ahead of time with the same `scripts/train.py` and recipe as Lab 2, and merges it into the base model on a **CPU-only** SageMaker training job (typically 10-20 minutes).

The job gets the same name prefix and output location as Lab 2, so [3-deployment.ipynb](./3-deployment.ipynb) finds and deploys it without changes. Lab 4 (evaluation) still reads the test split written by [1-prepare-data.ipynb](./1-prepare-data.ipynb), so run Lab 1 if you plan to evaluate.

## Prerequisites

### Setup and dependencies

In [ ]:
import boto3
from sagemaker.core.helper.session_helper import Session, get_execution_role

sess = Session()
sagemaker_session_bucket = None

if sagemaker_session_bucket is None and sess is not None:
    # set to default bucket if a bucket name is not given
    sagemaker_session_bucket = sess.default_bucket()

try:
    role = get_execution_role()
except ValueError:
    iam = boto3.client("iam")
    role = iam.get_role(RoleName="sagemaker_execution_role")["Role"]["Arn"]

s3_client = boto3.client("s3")
sm_client = boto3.client("sagemaker", region_name=sess.boto_region_name)
sess = Session(default_bucket=sagemaker_session_bucket)
bucket_name = sess.default_bucket()
default_prefix = sess.default_bucket_prefix

print(f"sagemaker role arn: {role}")
print(f"sagemaker bucket: {sess.default_bucket()}")
print(f"sagemaker session region: {sess.boto_region_name}")

In [ ]:
model_id = "Qwen/Qwen3-4B-Instruct-2507"

# Same prefix as Lab 2 and 3-deployment.ipynb
job_prefix = f"train-{model_id.split('/')[-1].replace('.', '-')}-sft"

***

## Upload the pre-trained adapter

`pretrained/adapter/` holds only the LoRA weights (about 66 MB). The base model is downloaded from Hugging Face inside the job.

In [ ]:
import os

adapter_dir = "./pretrained/adapter"
adapter_prefix = f"{(default_prefix + '/') if default_prefix else ''}pretrained/{job_prefix}/adapter"

for file_name in sorted(os.listdir(adapter_dir)):
    s3_client.upload_file(os.path.join(adapter_dir, file_name), bucket_name, f"{adapter_prefix}/{file_name}")
    print(f"uploaded {file_name}")

adapter_s3_uri = f"s3://{bucket_name}/{adapter_prefix}/"
adapter_s3_uri

***

## Merge the adapter on a CPU training job

[scripts_import/merge_adapter.py](./scripts_import/merge_adapter.py) loads the base model in bf16, applies the adapter with `peft`, calls `merge_and_unload()` and saves the result to `/opt/ml/model` exactly like `train.py` does after training: safetensors shards, `config.json`, `generation_config.json` and the tokenizer.

* `base_job_name` starts with `job_prefix`, so `get_last_job_name(job_prefix)` in 3-deployment.ipynb picks this job.
* `output_path` is the same as Lab 2, and the job writes a compressed `model.tar.gz`, like Lab 2.
* The merge holds the 8 GB bf16 model in memory; `ml.m5.4xlarge` (64 GB RAM) leaves plenty of headroom. Any CPU instance with at least 32 GB works if you have no quota for it (`ml.m5.4xlarge for training job usage`).

In [ ]:
from sagemaker.core import image_uris

instance_type = "ml.m5.4xlarge"

image_uri = image_uris.retrieve(
    framework="pytorch",
    region=sess.boto_region_name,
    version="2.8.0",
    instance_type=instance_type,
    image_scope="training"
)

image_uri

In [ ]:
from sagemaker.train.configs import (
    Compute,
    OutputDataConfig,
    SourceCode,
    StoppingCondition,
)
from sagemaker.train.model_trainer import ModelTrainer

source_code = SourceCode(
    source_dir="./scripts_import",
    requirements="requirements.txt",
    entry_script="merge_adapter.py",
)

compute_configs = Compute(
    instance_type=instance_type,
    instance_count=1,
    volume_size_in_gb=100,
    keep_alive_period_in_seconds=0,
)

# Same output location as Lab 2: 3-deployment.ipynb builds the model path from job_prefix
if default_prefix:
    output_path = f"s3://{bucket_name}/{default_prefix}/{job_prefix}"
else:
    output_path = f"s3://{bucket_name}/{job_prefix}"

environment = {"HF_HOME": "/opt/ml/input/hf_home"}  # cache the base model on the job's storage volume
if os.environ.get("HF_TOKEN"):
    environment["HF_TOKEN"] = os.environ["HF_TOKEN"]

model_trainer = ModelTrainer(
    training_image=image_uri,
    source_code=source_code,
    base_job_name=f"{job_prefix}-import",
    compute=compute_configs,
    stopping_condition=StoppingCondition(max_runtime_in_seconds=3600),
    hyperparameters={"model_id": model_id},
    environment=environment,
    output_data_config=OutputDataConfig(s3_output_path=output_path),
)

In [ ]:
from sagemaker.train.configs import InputData

adapter_input = InputData(
    channel_name="adapter",
    data_source=adapter_s3_uri,  # S3 path of the uploaded adapter
)

model_trainer.train(input_data_config=[adapter_input], wait=False)

### Wait for the job

Prints one status line per minute. The job log is in CloudWatch (`/aws/sagemaker/TrainingJobs`) if you need it.

In [ ]:
import time

job_name = sm_client.list_training_jobs(
    NameContains=f"{job_prefix}-import", SortBy="CreationTime", SortOrder="Descending", MaxResults=1
)["TrainingJobSummaries"][0]["TrainingJobName"]
print(f"job: {job_name}")

while True:
    desc = sm_client.describe_training_job(TrainingJobName=job_name)
    status = desc["TrainingJobStatus"]
    print(f"{time.strftime('%H:%M:%S')}  {status} / {desc['SecondaryStatus']}", flush=True)
    if status in ("Completed", "Failed", "Stopped"):
        break
    time.sleep(60)

if status != "Completed":
    raise RuntimeError(f"{job_name} ended {status}: {desc.get('FailureReason', '')}")

print(f"model artifacts: {desc['ModelArtifacts']['S3ModelArtifacts']}")
print(f"3-deployment.ipynb will use: {output_path}/{job_name}/output/model.tar.gz")

***

## Next step

Continue with [3-deployment.ipynb](./3-deployment.ipynb). Its `get_last_job_name(job_prefix)` returns the newest `Completed` job whose name starts with `train-Qwen3-4B-Instruct-2507-sft`, which is this import job (if you also completed Lab 2, the newer of the two jobs wins).